# Voice-Controlled Audio Player (Yes/No) — Phiên bản tối ưu

**Cải thiện so với v1:**
1. VAD (Voice Activity Detection) — cắt silence trước/sau
2. Data Augmentation — tăng 20 → 200+ mẫu
3. MFCC Normalization
4. Model nhỏ hơn, phù hợp dataset
5. Energy gating khi inference

In [1]:
import os
import time
import numpy as np
import librosa
import sounddevice as sd
import pygame

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Device: cuda


In [2]:
# ============================================================
# MFCC EXTRACTION VỚI VAD (Voice Activity Detection)
# ============================================================

MAX_PAD_LEN = 44  # ~1 giây ở sr=22050
N_MFCC = 40

def extract_mfcc(file_path_or_audio, sr=22050, trim_silence=True):
    """Trích xuất MFCC với VAD trim + normalization."""
    try:
        if isinstance(file_path_or_audio, str):
            audio, sr = librosa.load(file_path_or_audio, sr=sr)
        else:
            audio = file_path_or_audio
        
        # === VAD: Cắt bỏ silence trước/sau ===
        if trim_silence:
            audio, _ = librosa.effects.trim(audio, top_db=20)
        
        # Bỏ qua nếu audio quá ngắn (< 0.1s)
        if len(audio) < sr * 0.1:
            return None
        
        # Trích xuất MFCC
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        
        # === Chuẩn hóa MFCC (zero-mean, unit-variance) ===
        mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-8)
        
        # Pad hoặc truncate
        if mfcc.shape[1] < MAX_PAD_LEN:
            pad_width = MAX_PAD_LEN - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :MAX_PAD_LEN]
        
        return mfcc.T  # (timesteps, features)
    except Exception as e:
        print(f"Lỗi: {e}")
        return None

print("Hàm extract_mfcc đã sẵn sàng (có VAD + normalize).")

Hàm extract_mfcc đã sẵn sàng (có VAD + normalize).


In [3]:
# ============================================================
# DATA AUGMENTATION — Tăng dữ liệu từ 20 → 200+ mẫu
# ============================================================

def augment_audio(audio, sr=22050):
    """Tạo nhiều biến thể từ 1 audio gốc."""
    augmented = []
    
    # 1. Bản gốc (đã trim)
    trimmed, _ = librosa.effects.trim(audio, top_db=20)
    augmented.append(trimmed)
    
    # 2. Thêm white noise (3 mức)
    for noise_factor in [0.005, 0.01, 0.02]:
        noise = noise_factor * np.random.randn(len(trimmed))
        augmented.append(trimmed + noise.astype(np.float32))
    
    # 3. Time shift (dịch trái/phải)
    for shift in [int(sr*0.05), int(-sr*0.05), int(sr*0.1)]:
        shifted = np.roll(trimmed, shift)
        augmented.append(shifted)
    
    # 4. Speed change
    for rate in [0.85, 1.15]:
        stretched = librosa.effects.time_stretch(trimmed, rate=rate)
        augmented.append(stretched)
    
    # 5. Pitch shift
    for n_steps in [-2, 2]:
        pitched = librosa.effects.pitch_shift(trimmed, sr=sr, n_steps=n_steps)
        augmented.append(pitched)
    
    # 6. Volume change
    for gain in [0.7, 1.3]:
        augmented.append(trimmed * gain)
    
    return augmented  # 13 biến thể từ 1 file gốc

print("Augmentation functions ready. Mỗi file gốc → 13 biến thể.")
print(f"Dự kiến: 10 files × 13 aug × 2 classes = 260 mẫu training.")

Augmentation functions ready. Mỗi file gốc → 13 biến thể.
Dự kiến: 10 files × 13 aug × 2 classes = 260 mẫu training.


In [4]:
# ============================================================
# MODEL — Nhỏ gọn hơn, phù hợp dataset nhỏ
# ============================================================

class SpeechLSTM(nn.Module):
    def __init__(self, input_size=40, hidden_size=32, num_classes=2):
        super(SpeechLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2,
                           batch_first=True, dropout=0.3, bidirectional=True)
        self.bn = nn.BatchNorm1d(hidden_size * 2)  # *2 vì bidirectional
        self.fc1 = nn.Linear(hidden_size * 2, 16)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(16, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # Lấy timestep cuối
        out = self.bn(out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

print("Model SpeechLSTM: 2-layer Bidirectional LSTM (hidden=32) + BN")
model = SpeechLSTM().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Model SpeechLSTM: 2-layer Bidirectional LSTM (hidden=32) + BN
Total parameters: 45,234


In [ ]:
# ============================================================
# LOAD DATA + AUGMENT + TRAIN
# ============================================================

DATASET_PATH = r"dataset/"
SR = 22050
X, y = [], []

for label_name in ['no', 'yes']:
    label_idx = 0 if label_name == 'no' else 1
    folder_path = os.path.join(DATASET_PATH, label_name)
    
    if not os.path.exists(folder_path):
        print(f"Không tìm thấy thư mục: {folder_path}")
        continue
    
    file_count = 0
    for file_name in os.listdir(folder_path):
        if not file_name.endswith('.wav'):
            continue
        file_path = os.path.join(folder_path, file_name)
        audio, sr = librosa.load(file_path, sr=SR)
        
        # Tạo các biến thể augmented
        aug_audios = augment_audio(audio, sr=SR)
        
        for aug_audio in aug_audios:
            features = extract_mfcc(aug_audio, sr=SR, trim_silence=False)  # Đã trim trong augment
            if features is not None:
                X.append(features)
                y.append(label_idx)
                file_count += 1
    
    print(f"  [{label_name.upper()}] Tạo được {file_count} mẫu (gốc + augmented)")

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)
print(f"\nTổng dataset: {len(X)} mẫu — X shape: {X.shape}, y shape: {y.shape}")
print(f"  No: {np.sum(y==0)}, Yes: {np.sum(y==1)}")

# Chia Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                     random_state=42, stratify=y)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Train
model = SpeechLSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

EPOCHS = 30
best_acc = 0.0
print(f"\nBắt đầu training {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    
    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = train_loss / len(train_loader.dataset)
    accuracy = correct / total
    scheduler.step(epoch_loss)
    
    # Lưu model tốt nhất
    if accuracy > best_acc:
        best_acc = accuracy
        torch.save(model.state_dict(), "speech_control_v2.pth")
    
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {epoch_loss:.4f} — Acc: {accuracy*100:.1f}% — Best: {best_acc*100:.1f}%")

print(f"\nTraining xong! Best accuracy: {best_acc*100:.1f}%")
print("Đã lưu model tốt nhất vào speech_control_v2.pth")

  [NO] Tạo được 130 mẫu (gốc + augmented)
  [YES] Tạo được 130 mẫu (gốc + augmented)

Tổng dataset: 260 mẫu — X shape: (260, 44, 40), y shape: (260,)
  No: 130, Yes: 130
Train: 208, Test: 52

Bắt đầu training 30 epochs...
Epoch 1/30 — Loss: 0.7189 — Acc: 50.0% — Best: 50.0%
Epoch 10/30 — Loss: 0.7035 — Acc: 50.0% — Best: 69.2%
Epoch 20/30 — Loss: 0.2584 — Acc: 100.0% — Best: 100.0%
Epoch 30/30 — Loss: 0.0156 — Acc: 100.0% — Best: 100.0%

✅ Training xong! Best accuracy: 100.0%
Đã lưu model tốt nhất vào speech_control_v2.pth


In [20]:
# ============================================================
# ĐÁNH GIÁ MODEL — Confusion Matrix + Test từng file gốc
# ============================================================
DATASET_PATH = r'dataset/'
model.load_state_dict(torch.load("speech_control_v2.pth", map_location=device))
model.eval()

# Confusion matrix trên test set
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("=== Classification Report ===")
print(classification_report(all_labels, all_preds, target_names=['No', 'Yes']))

cm = confusion_matrix(all_labels, all_preds)
print(f"Confusion Matrix:\n{cm}")
print(f"  [TN={cm[0,0]}, FP={cm[0,1]}]")
print(f"  [FN={cm[1,0]}, TP={cm[1,1]}]")

# Test từng file gốc (không augmented)
print("\n=== Test trên từng file GỐC ===")
for label_name in ['no', 'yes']:
    folder_path = os.path.join(DATASET_PATH, label_name)
    for file_name in sorted(os.listdir(folder_path)):
        if not file_name.endswith('.wav'):
            continue
        file_path = os.path.join(folder_path, file_name)
        features = extract_mfcc(file_path, sr=SR, trim_silence=True)
        if features is None:
            continue
        
        inp = torch.from_numpy(features).unsqueeze(0).to(torch.float32).to(device)
        with torch.no_grad():
            out = model(inp)
            probs = torch.softmax(out, dim=1)
            conf, pred = torch.max(probs, 1)
        
        pred_label = 'YES' if pred.item() == 1 else 'NO'
        status = '✅' if pred_label == label_name.upper() else '❌'
        print(f"  {status} {label_name}/{file_name} → {pred_label} ({conf.item()*100:.1f}%)")

=== Classification Report ===
              precision    recall  f1-score   support

          No       1.00      1.00      1.00        26
         Yes       1.00      1.00      1.00        26

    accuracy                           1.00        52
   macro avg       1.00      1.00      1.00        52
weighted avg       1.00      1.00      1.00        52

Confusion Matrix:
[[26  0]
 [ 0 26]]
  [TN=26, FP=0]
  [FN=0, TP=26]

=== Test trên từng file GỐC ===
  ✅ no/Recording (10).wav → NO (98.6%)
  ✅ no/Recording (2).wav → NO (98.8%)
  ✅ no/Recording (3).wav → NO (97.8%)
  ✅ no/Recording (4).wav → NO (97.8%)
  ✅ no/Recording (5).wav → NO (98.1%)
  ✅ no/Recording (6).wav → NO (98.4%)
  ✅ no/Recording (7).wav → NO (98.6%)
  ✅ no/Recording (8).wav → NO (98.7%)
  ✅ no/Recording (9).wav → NO (98.1%)
  ✅ no/Recording.wav → NO (98.1%)
  ✅ yes/Recording (10).wav → YES (98.5%)
  ✅ yes/Recording (2).wav → YES (98.5%)
  ✅ yes/Recording (3).wav → YES (98.0%)
  ✅ yes/Recording (4).wav → YES (97.9%)
  ✅

In [24]:
# ============================================================
# INFERENCE — Điều khiển audio bằng giọng nói
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# Load model tốt nhất
model = SpeechLSTM().to(device)
model.load_state_dict(torch.load("speech_control_v2.pth", map_location=device))
model.eval()

# Pygame
AUDIO_FILE = r"C:\Users\Admin\Desktop\SLP301\practice_analysis\Walen - Midnight Swing (freetouse.com).wav"
pygame.mixer.init()
pygame.mixer.music.load(AUDIO_FILE)

is_playing = False
has_started = False

# UI
play_btn = widgets.Button(description=' ⏸ TẮT', button_style='danger',
                          layout=widgets.Layout(width='150px', height='40px'))
energy_bar = widgets.FloatProgress(value=0, min=0, max=0.15, description='Energy:',
                                   bar_style='info', layout=widgets.Layout(width='300px'))
status_info = widgets.HTML(
    value="<span style='font-size:16px; margin-left:15px; color:gray;'>"
          "Chờ... Nói <b>'Yes'</b> để phát.</span>")
log_output = widgets.Output(layout=widgets.Layout(max_height='200px', overflow='auto'))

ui = widgets.VBox([
    widgets.HBox([play_btn, energy_bar, status_info],
                 layout=widgets.Layout(align_items='center', margin='10px 0')),
    log_output
])
clear_output()
display(ui)

# === Vòng lặp nghe ===
FS = 22050
DURATION = 1.0
CONFIDENCE_THRESHOLD = 0.85
ENERGY_THRESHOLD = 0.015  # Ngưỡng năng lượng tối thiểu để coi là "có tiếng nói"

try:
    while True:
        recording = sd.rec(int(DURATION * FS), samplerate=FS, channels=1, dtype='float32')
        sd.wait()
        audio_data = recording.flatten()
        
        # === ENERGY GATING: Bỏ qua nếu im lặng ===
        energy = float(np.sqrt(np.mean(audio_data**2)))
        energy_bar.value = min(energy, 0.15)
        
        if energy < ENERGY_THRESHOLD:
            # Im lặng → không predict, tránh bias
            time.sleep(0.05)
            continue
        
        # VAD trim trước khi extract MFCC
        features = extract_mfcc(audio_data, sr=FS, trim_silence=True)
        if features is None:
            continue
        
        input_tensor = torch.from_numpy(features).unsqueeze(0).to(torch.float32).to(device)
        
        with torch.no_grad():
            outputs = model(input_tensor)
            probs = torch.softmax(outputs, dim=1)
            confidence, predicted_class = torch.max(probs, 1)
            predicted_class = predicted_class.item()
            confidence = confidence.item()
        
        if confidence > CONFIDENCE_THRESHOLD:
            if predicted_class == 1 and not is_playing:  # YES → BẬT
                if not has_started:
                    pygame.mixer.music.play(-1)
                    has_started = True
                else:
                    pygame.mixer.music.unpause()
                is_playing = True
                play_btn.description = ' ▶ ĐANG PHÁT'
                play_btn.button_style = 'success'
                status_info.value = (
                    f"<span style='font-size:16px; margin-left:15px; color:#28a745;'>"
                    f"<b>YES ({confidence*100:.1f}%)</b> — Audio đang phát...</span>")
                with log_output:
                    print(f"BẬT — YES ({confidence*100:.1f}%) energy={energy:.4f}")

            elif predicted_class == 0 and is_playing:  # NO → TẮT
                pygame.mixer.music.pause()
                is_playing = False
                play_btn.description = ' ⏸ TẮT'
                play_btn.button_style = 'danger'
                status_info.value = (
                    f"<span style='font-size:16px; margin-left:15px; color:#dc3545;'>"
                    f"<b> NO ({confidence*100:.1f}%)</b> — Đã tắt. Nói 'Yes' để phát lại.</span>")
                with log_output:
                    print(f"⏸ TẮT — NO ({confidence*100:.1f}%) energy={energy:.4f}")

        time.sleep(0.05)

except KeyboardInterrupt:
    pygame.mixer.music.stop()
    status_info.value = "<span style='font-size:16px; margin-left:15px; color:gray;'>Đã dừng.</span>"
    with log_output:
        print("Đã dừng chương trình.")